<a href="https://colab.research.google.com/github/kraftmasch/OrchIso-Two_Pass_Orchestral_Stem_Separator/blob/main/OrchIso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Final Project KCVanguard**

#**OrchIso**

by: tembok ratapan solo

In [ ]:
!pip install -q demucs
!pip install -q torch torchaudio
!apt-get install -y -q ffmpeg

print('Dependency berhasil diinstall')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.3/249.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 5.0 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Dependency berhasil diinstall


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DATASET = '/content/drive/MyDrive/orchiso_dataset'
DIRTY_DIR     = f'{DRIVE_DATASET}/dirty'
CLEAN_DIR     = f'{DRIVE_DATASET}/clean'

AUDIO_EXTS = ('.mp3', '.wav', '.flac')

dirty_files = sorted([f for f in os.listdir(DIRTY_DIR) if f.lower().endswith(AUDIO_EXTS)])
clean_files = sorted([f for f in os.listdir(CLEAN_DIR) if f.lower().endswith(AUDIO_EXTS)])
paired      = [f for f in dirty_files if f in set(clean_files)]
only_dirty  = [f for f in dirty_files if f not in set(clean_files)]
only_clean  = [f for f in clean_files if f not in set(dirty_files)]

print(f'Jumlah pasangan valid   : {len(paired)} file')
if only_dirty:
    print(f'File hanya ada di dirty : {only_dirty}')
if only_clean:
    print(f'File hanya ada di clean : {only_clean}')

if len(paired) == 0:
    raise RuntimeError('Pasangan file tidak ditemukan')

Mounted at /content/drive
Jumlah pasangan valid   : 30 file


In [ ]:
import torch
import torch.nn as nn
import torchaudio
import numpy as np
from torch.utils.data import Dataset, DataLoader

SAMPLE_RATE     = 44100
CHUNK_SECONDS   = 4
CHUNK_SIZE      = SAMPLE_RATE * CHUNK_SECONDS

N_FFT       = 2048
HOP_LENGTH  = 512
WIN_LENGTH  = 2048

CHUNKS_PER_FILE = 30
BATCH_SIZE      = 6
EPOCHS          = 30
LR              = 1e-3
EARLY_STOP      = 7

class ConvBlock2D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class SpectrogramUNet(nn.Module):
    def __init__(self, features=[16, 32, 64, 128]):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        self.enc1 = ConvBlock2D(2,           features[0])
        self.enc2 = ConvBlock2D(features[0], features[1])
        self.enc3 = ConvBlock2D(features[1], features[2])
        self.enc4 = ConvBlock2D(features[2], features[3])

        self.bottleneck = ConvBlock2D(features[3], features[3] * 2)

        self.up4  = nn.ConvTranspose2d(features[3]*2, features[3], kernel_size=2, stride=2)
        self.dec4 = ConvBlock2D(features[3]*2, features[3])
        self.up3  = nn.ConvTranspose2d(features[3], features[2], kernel_size=2, stride=2)
        self.dec3 = ConvBlock2D(features[2]*2, features[2])
        self.up2  = nn.ConvTranspose2d(features[2], features[1], kernel_size=2, stride=2)
        self.dec2 = ConvBlock2D(features[1]*2, features[1])
        self.up1  = nn.ConvTranspose2d(features[1], features[0], kernel_size=2, stride=2)
        self.dec1 = ConvBlock2D(features[0]*2, features[0])

        self.final = nn.Sequential(
            nn.Conv2d(features[0], 2, kernel_size=1),
            nn.Sigmoid()
        )

    def _cat(self, up, skip):
        h = min(up.shape[2], skip.shape[2])
        w = min(up.shape[3], skip.shape[3])
        return torch.cat([up[:, :, :h, :w], skip[:, :, :h, :w]], dim=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(self._cat(self.up4(b),  e4))
        d3 = self.dec3(self._cat(self.up3(d4), e3))
        d2 = self.dec2(self._cat(self.up2(d3), e2))
        d1 = self.dec1(self._cat(self.up1(d2), e1))
        return self.final(d1)


def to_spectrogram(wav, n_fft=N_FFT, hop=HOP_LENGTH, win=WIN_LENGTH):
    window  = torch.hann_window(win).to(wav.device)
    results = []
    phases  = []
    for ch in range(wav.shape[0]):
        stft = torch.stft(
            wav[ch], n_fft=n_fft, hop_length=hop,
            win_length=win, window=window,
            return_complex=True
        )
        results.append(stft.abs())
        phases.append(stft.angle())
    return torch.stack(results), torch.stack(phases)


def to_waveform(magnitude, phase, length,
                n_fft=N_FFT, hop=HOP_LENGTH, win=WIN_LENGTH):
    window  = torch.hann_window(win).to(magnitude.device)
    channels = []
    for ch in range(magnitude.shape[0]):
        stft_complex = torch.polar(magnitude[ch], phase[ch])
        wav = torch.istft(
            stft_complex, n_fft=n_fft, hop_length=hop,
            win_length=win, window=window, length=length
        )
        channels.append(wav)
    return torch.stack(channels)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device   : {device}')
model = SpectrogramUNet().to(device)
print(f'Parameter: {sum(p.numel() for p in model.parameters()):,}')

dummy = torch.randn(2, CHUNK_SIZE)
mag, phase = to_spectrogram(dummy)
print(f'Spectrogram shape: {mag.shape}  (channels, freq_bins, time_frames)')

Device   : cpu
Parameter: 1,943,922
Spectrogram shape: torch.Size([2, 1025, 345])  (channels, freq_bins, time_frames)


In [ ]:
class SpectrogramDataset(Dataset):
    def __init__(self, dirty_dir, clean_dir, files,
                 chunk_size=CHUNK_SIZE, chunks_per_file=CHUNKS_PER_FILE):
        self.dirty_dir       = dirty_dir
        self.clean_dir       = clean_dir
        self.files           = files
        self.chunk_size      = chunk_size
        self.chunks_per_file = chunks_per_file
        total = len(files) * chunks_per_file
        print(f'{len(files)} file × {chunks_per_file} chunk = {total} sampel')

    def _load(self, path):
        wav, sr = torchaudio.load(path)
        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
        if wav.shape[0] == 1:
            wav = wav.repeat(2, 1)
        return wav[:2]

    def __len__(self):
        return len(self.files) * self.chunks_per_file

    def __getitem__(self, idx):
        fname = self.files[idx % len(self.files)]
        dirty = self._load(os.path.join(self.dirty_dir, fname))
        clean = self._load(os.path.join(self.clean_dir, fname))

        min_len = min(dirty.shape[1], clean.shape[1])
        dirty, clean = dirty[:, :min_len], clean[:, :min_len]

        max_start = max(min_len - self.chunk_size, 1)
        start = torch.randint(0, max_start, (1,)).item()
        d = dirty[:, start:start + self.chunk_size]
        c = clean[:, start:start + self.chunk_size]

        if d.shape[1] < self.chunk_size:
            pad = self.chunk_size - d.shape[1]
            d = torch.nn.functional.pad(d, (0, pad))
            c = torch.nn.functional.pad(c, (0, pad))

        d_mag, _ = to_spectrogram(d)
        c_mag, _ = to_spectrogram(c)

        d_mag = torch.log1p(d_mag)
        c_mag = torch.log1p(c_mag)

        return d_mag, c_mag

In [ ]:
def train_model(model, dirty_dir, clean_dir, paired_files):
    import random
    files = paired_files.copy()
    random.shuffle(files)

    split       = int(len(files) * 0.8)
    train_files = files[:split]
    val_files   = files[split:] if split < len(files) else files[:2]

    print(f'Train: {len(train_files)} file | Val: {len(val_files)} file\n')

    train_ds = SpectrogramDataset(dirty_dir, clean_dir, train_files)
    val_ds   = SpectrogramDataset(dirty_dir, clean_dir, val_files,
                                   chunks_per_file=5)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5
    )
    criterion = nn.L1Loss()

    best_val   = float('inf')
    patience_c = 0
    prev_lr    = LR

    print(f'Training {EPOCHS} epoch (early stop {EARLY_STOP}x)...\n')
    print(f'{"Epoch":>5} | {"Train":>10} | {"Val":>10} |')
    print('-' * 33)

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for d_spec, c_spec in train_loader:
            d_spec = d_spec.to(device)
            c_spec = c_spec.to(device)

            optimizer.zero_grad()

            mask = model(d_spec)

            h = min(mask.shape[2], c_spec.shape[2])
            w = min(mask.shape[3], c_spec.shape[3])

            pred = mask[:, :, :h, :w] * d_spec[:, :, :h, :w]
            tgt  = c_spec[:, :, :h, :w]

            loss = criterion(pred, tgt)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for d_spec, c_spec in val_loader:
                d_spec = d_spec.to(device)
                c_spec = c_spec.to(device)
                mask   = model(d_spec)
                h = min(mask.shape[2], c_spec.shape[2])
                w = min(mask.shape[3], c_spec.shape[3])
                pred = mask[:, :, :h, :w] * d_spec[:, :, :h, :w]
                tgt  = c_spec[:, :, :h, :w]
                val_loss += criterion(pred, tgt).item()

        avg_t = train_loss / len(train_loader)
        avg_v = val_loss   / len(val_loader)
        scheduler.step(avg_v)

        curr_lr = optimizer.param_groups[0]['lr']
        lr_tag  = f' | LR→{curr_lr:.1e}' if curr_lr != prev_lr else ''
        prev_lr = curr_lr

        tag = ''
        if avg_v < best_val:
          best_val       = avg_v
          best_epoch     = epoch + 1
          best_state     = {k: v.clone() for k, v in model.state_dict().items()}
          patience_c     = 0
          tag = f'← best (epoch {best_epoch})'
        else:
            patience_c += 1
            if patience_c >= EARLY_STOP:
                print(f'{epoch+1:>5} | {avg_t:>10.5f} | {avg_v:>10.5f} | early stop ⏹️')
                break
            tag = f'patience {patience_c}'

        print(f'{epoch+1:>5} | {avg_t:>10.5f} | {avg_v:>10.5f} | {tag}{lr_tag}')

    torch.save(model.state_dict(), 'orchestra_model_last.pth')
    print(f'\nComplete \n Best epoch: {best_epoch} | Val loss: {best_val:.5f}')

    import shutil
    shutil.copy('orchestra_model_best.pth',
                f'{DRIVE_DATASET}/orchestra_spectrogram_best.pth')
    print('Model tersimpan di GDrive')


train_model(model, DIRTY_DIR, CLEAN_DIR, paired)

Train: 24 file | Val: 6 file

📂 24 file × 30 chunk = 720 sampel
📂 6 file × 5 chunk = 30 sampel
🚀 Training 30 epoch (early stop 7x)...

Epoch |      Train |        Val |
------------------------------------
    1 |    0.10536 |    0.08568 | ← best 💾
    2 |    0.09589 |    0.08350 | ← best 💾
    3 |    0.08794 |    0.09889 | patience 1
    4 |    0.08882 |    0.08369 | patience 2
    5 |    0.08342 |    0.07669 | ← best 💾
    6 |    0.08336 |    0.06922 | ← best 💾
    7 |    0.08150 |    0.07408 | patience 1
    8 |    0.07925 |    0.07713 | patience 2
    9 |    0.07539 |    0.07474 | patience 3
   10 |    0.07505 |    0.09405 | patience 4 | LR→5.0e-04
   11 |    0.07264 |    0.07004 | patience 5
   12 |    0.07193 |    0.07377 | patience 6
   13 |    0.07237 |    0.06028 | ← best 💾
   14 |    0.07172 |    0.06383 | patience 1
   15 |    0.07164 |    0.07933 | patience 2
   16 |    0.07213 |    0.07822 | patience 3
   17 |    0.06904 |    0.07719 | patience 4 | LR→2.5e-04
   18 |    0.

In [ ]:
import os
from google.colab import files
import subprocess

uploaded  = files.upload()
song_path = list(uploaded.keys())[0]
print(f'File Uploaded: {song_path}')

DEMUCS_MODEL = 'mdx_extra'

subprocess.run([
    'python', '-m', 'demucs',
    '--name', DEMUCS_MODEL,
    '--out', 'separated',
    song_path,
], check=True)

song_name  = os.path.splitext(os.path.basename(song_path))[0]
stems_dir  = f'separated/{DEMUCS_MODEL}/{song_name}'

other_stem = f'{stems_dir}/other.wav'
print(f'\nInput ke model: {other_stem}')

Saving Thursday Morning.mp3 to Thursday Morning.mp3
File Uploaded: Thursday Morning.mp3

Input ke model: separated/mdx_extra/Thursday Morning/other.wav


In [ ]:
def process_audio(model, input_path, output_path):
    model.eval()

    wav, sr = torchaudio.load(input_path)
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
    if wav.shape[0] == 1:
        wav = wav.repeat(2, 1)
    wav = wav[:2]

    total   = wav.shape[1]
    hop     = CHUNK_SIZE // 2
    out_buf = torch.zeros_like(wav)
    w_buf   = torch.zeros(total)
    window  = torch.hann_window(CHUNK_SIZE)
    starts  = list(range(0, total - CHUNK_SIZE + 1, hop))

    griffin_lim = torchaudio.transforms.GriffinLim(
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        power=1.0,
        n_iter=32,
    )

    print(f'Memproses {total / SAMPLE_RATE:.1f} detik | {len(starts)} chunk...')

    with torch.no_grad():
        for i, start in enumerate(starts):
            chunk = wav[:, start:start + CHUNK_SIZE]

            mag, phase = to_spectrogram(chunk)
            mag_log    = torch.log1p(mag).unsqueeze(0).to(device)

            mask = model(mag_log).squeeze(0).cpu()

            f_min = min(mask.shape[1], mag.shape[1])
            w_min = min(mask.shape[2], mag.shape[2])

            masked_mag = mask[:, :f_min, :w_min] * mag[:, :f_min, :w_min]

            expected_f = N_FFT // 2 + 1
            if masked_mag.shape[1] < expected_f:
                pad_f      = expected_f - masked_mag.shape[1]
                masked_mag = torch.nn.functional.pad(masked_mag, (0, 0, 0, pad_f))

            channels = []
            for ch in range(masked_mag.shape[0]):
                recon = griffin_lim(masked_mag[ch].unsqueeze(0))
                channels.append(recon.squeeze(0))
            out_chunk = torch.stack(channels)

            n = min(out_chunk.shape[1], CHUNK_SIZE)

            out_buf[:, start:start + n] += out_chunk[:, :n] * window[:n].unsqueeze(0)
            w_buf[start:start + n]      += window[:n]

            if (i + 1) % 20 == 0:
                print(f'  {i+1}/{len(starts)} chunk selesai...')

    w_buf   = w_buf.clamp(min=1e-8)
    out_buf = (out_buf / w_buf.unsqueeze(0)).clamp(-1.0, 1.0)

    os.makedirs('output', exist_ok=True)
    torchaudio.save(output_path, out_buf, SAMPLE_RATE)
    print(f'\nTersimpan pada: {output_path}')

model.load_state_dict(torch.load('orchestra_model_best.pth'))

safe_name   = song_name.replace(' ', '_')
output_path = f'output/{safe_name}_orchiso.wav'
process_audio(model, other_stem, output_path)

FileNotFoundError: [Errno 2] No such file or directory: 'orchestra_model_best.pth'

In [ ]:
from google.colab import files
import shutil

files.download(output_path)
print('Download dimulai')

✅ Tersimpan di Drive: /content/drive/MyDrive/orchiso_dataset/Thursday Morning (1)_orchestra_v3.wav


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Download dimulai!
